# 02 · Typed — ontology-guided extraction

cognify was run with a domain ontology (`ontology.ttl`), so the graph follows that vocabulary: ontology classes become `EntityType` nodes and matched entities are flagged `ontology_valid`.

> Run `build.py` first.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
from _common import config
config.require_openai_key(); config.quiet()          # quiet cognee's verbose logs
config.configure("cognee_typed")
import cognee
from cognee.modules.search.types import SearchType
from cognee.infrastructure.databases.graph import get_graph_engine

def _name(node):
    if isinstance(node, dict):
        return str(node.get("name") or (node.get("text") or "")[:40] or node.get("id") or "?")
    return str(node)[:40]

def render(results):
    if isinstance(results, (str, bytes)) or not isinstance(results, (list, tuple)):
        results = [results] if results else []
    for r in results[:4]:
        if isinstance(r, (tuple, list)) and len(r) == 3:
            src, edge, tgt = r
            rel = edge.get("relationship_name") if isinstance(edge, dict) else str(edge)
            print(f"   ({_name(src)}) -[{rel}]-> ({_name(tgt)})")
        elif isinstance(r, dict):
            print("   " + str(r.get("text") or r.get("name") or r)[:150])
        else:
            print("   " + str(r).strip().replace(chr(10), " ")[:550])
nodes, _ = await (await get_graph_engine()).get_graph_data()
entity_types = {p.get("name") for _, p in nodes if p.get("type") == "EntityType"}
aligned = [p.get("name") for _, p in nodes if p.get("ontology_valid") and p.get("type") == "Entity"]
print("EntityType nodes total:", len(entity_types))
print("entities aligned to the ontology (ontology_valid):", len(aligned))
print("aligned:", ", ".join(map(str, aligned[:12])))

EntityType nodes total: 235
entities aligned to the ontology (ontology_valid): 22
aligned: alaska, albania, allan_dwan, alain_connes, atlantic_ocean, anthropology, anarchism, alabama, ayn_rand, america_the_beautiful, aristotle, actrius


## Typed triplets (`INSIGHTS`) and a grounded answer

In [2]:
q = f"What is {aligned[0]} and what is it connected to?" if aligned else "What are the key entities?"
print("Q:", q, "\n")
render(await config.search(query_text=q, query_type=SearchType.INSIGHTS))
print()
render(await config.search(query_text=q, query_type=SearchType.GRAPH_COMPLETION))

Q: What is alaska and what is it connected to? 



   (alaska) -[borders]-> (canada)
   (alaska) -[served_as_entry_point_for_settlement]-> (bering land bridge)
   (alaska) -[most_population_north_of]-> (60th parallel)
   (alaska) -[shares_maritime_border_with]-> (bering strait)




AgensGraph statement failed: 42P01: relation "FunctionDefinition_source_code" does not exist



AgensGraph statement failed: 42P01: relation "ClassDefinition_source_code" does not exist



AgensGraph statement failed: 42P01: relation "CodeFile_name" does not exist


...


AgensGraph statement failed: 42P01: relation "TableRow_properties" does not exist



AgensGraph statement failed: 42P01: relation "ColumnValue_properties" does not exist



AgensGraph statement failed: 42P01: relation "CodeSummary_text" does not exist



AgensGraph statement failed: 42P01: relation "IndexSchema_text" does not exist


   Alaska is a non-contiguous U.S. state located on the northwest extremity of North America. It is connected to British Columbia and Yukon in Canada to the east, and it shares a maritime border with Russia's Chukotka Autonomous Okrug to the west across the Bering Strait. Alaska is also connected historically as an entry point for the initial settlement of North America via the Bering land bridge.
